# 05 · Çoklu seed: en iyi 3 modelin seed 123 ve 2026 tekrarları

**Önce GPU açın** (Colab: T4 GPU). Optuna'da bulunan en iyi parametrelerle ConvNeXt-Tiny, DeiT-Small ve ResNet50 iki yeni seed ile (123, 2026) tam eğitim kümesinde eğitilir; seed 42 zaten vardır.

- Veri bölmesi sabit; yalnızca ağırlık başlatma, veri sırası ve augmentation rastgeleliği değişir
- 6 koşu, toplam yaklaşık 50 dakika. Biten koşular atlanır, oturum kopsa aynı notebook'u yeniden çalıştırın
- Özet yalnızca **validation** kullanır. Test kümesi yüklenmez
- "En iyi mevcut model" kuralı: 3 seed ortalama validation macro F1'i en yüksek model

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
# Kod ve çıktılar Drive'da tek klasörde durur; ham veri oturum diskinde (/content/data) tutulur.
PROJECT = next((p for p in ('/content/drive/MyDrive/YZProje', '/content/drive/MyDrive/YZProje/YZProje')
                if os.path.isfile(p + '/src/config.py')), None)
assert PROJECT, "Drive'da YZProje/src/config.py bulunamadı: proje klasörünü MyDrive/YZProje olarak yerleştirin."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = PROJECT + '/outputs'
os.environ['KNEE_DATA_ROOT'] = '/content/data'
print('Proje klasörü:', PROJECT)

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0 timm==1.0.30 optuna==5.0.0
import torch
assert torch.cuda.is_available(), 'GPU yok! Colab: Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU | Kaggle: sağ panel Session options > Accelerator > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Görüntüler oturum diskinde yoksa: önce Drive'daki arşiv kopyası, o da yoksa bağlantıdan indirip arşivi Drive'a kaydeder.
from src import config as C, data_prep
print('Veri hazır:', data_prep.ensure_data())

## Seed 123 ve 2026 koşuları

In [ ]:
from src import optimize as O, seeds as SD
import pandas as pd
KEYS = O.top_candidates(3); print('Modeller:', KEYS)
SD.run_seeds(KEYS, seeds=(123, 2026))

## Özet (validation, 3 seed ortalaması ± std)

In [ ]:
import json
df = SD.summarize(KEYS)
show = df[['model','n_seeds','val_macro_f1_mean','val_macro_f1_std','val_balanced_accuracy_mean','val_qwk_mean',
           'val_qwk_std','val_mae_grade_mean','val_accuracy_mean']]
display(show.round(4))
print(json.load(open(C.OUTPUT_DIR / 'best_existing_model.json'))['best_existing_model'])

`outputs/results.csv`, `outputs/seed_summary.csv` ve `outputs/best_existing_model.json` dosyalarını paylaşın.